In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import os

# Veri yollarını tanımla
data_dir = "../data"
files = {
    "BIST100": "bist100_weekly.csv",
    "VIX": "vix_weekly.csv",
    "GOLD_TL": "gold_tl_weekly.csv",
    "BOND_2Y": "bond_2y_weekly.csv"
}

def load_and_clean(file_name, col_name):
    path = os.path.join(data_dir, file_name)
    
    # Türkçe/İngilizce CSV ayracını otomatik bulması için sep=None kullanıyoruz
    df = pd.read_csv(path, sep=None, engine='python')
    df = df.iloc[:, [0, 1]].copy() # İlk iki sütunu (Tarih ve Fiyat) güvenlice al
    df.columns = ["Date", col_name]
    
    print(f"--- {col_name} ---")
    print(f"1. Başlangıç Satır Sayısı: {len(df)}")
    print(f"2. Örnek Tarih: '{df['Date'].iloc[0]}' | Örnek Fiyat: '{df[col_name].iloc[0]}'")
    
    # Fiyat Sütunu Temizliği (Hem Türkçe hem İngilizce ondalıkları otomatik çözer)
    if df[col_name].dtype == object:
        df[col_name] = df[col_name].astype(str)
        # Türkçe formatı İngilizce formata zorla (Örn: 1.234,56 -> 1234.56)
        if df[col_name].str.contains(',').any() and df[col_name].str.contains('\.').any():
            df[col_name] = df[col_name].str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
        else:
            df[col_name] = df[col_name].str.replace(',', '.', regex=False)
            
        df[col_name] = pd.to_numeric(df[col_name], errors='coerce')
    
    # Tarih Sütunu Temizliği (Türkçe aylar varsa düzeltir)
    aylar = {"Oca": "Jan", "Şub": "Feb", "Mar": "Mar", "Nis": "Apr", "May": "May", "Haz": "Jun",
             "Tem": "Jul", "Ağu": "Aug", "Eyl": "Sep", "Eki": "Oct", "Kas": "Nov", "Ara": "Dec"}
    for tr, en in aylar.items():
        df["Date"] = df["Date"].astype(str).str.replace(tr, en, regex=False)
        
    df["Date"] = pd.to_datetime(df["Date"], format='mixed', dayfirst=True, errors='coerce')
    
    # Okunamayan tarih veya sayıları temizle ve tarihe göre diz
    df = df.dropna(subset=["Date", col_name])
    df = df.set_index("Date").sort_index()
    print(f"3. Temizlik Sonrası Satır Sayısı: {len(df)}\n")
    return df

# 1. Yükle ve Temizle
dfs = []
for key, file in files.items():
    dfs.append(load_and_clean(file, key))

# 2. Hepsini tarih atlamadan Dış Birleştirme (Outer Join) ile birleştir
df_merged = pd.concat(dfs, axis=1, sort=True)
print(f"-> Tüm veriler birleşti. Toplam Satır: {len(df_merged)}")

# 3. Haftalık (Cuma kapanışı) hizala ve eksik günleri önceki günlerin fiyatıyla (ffill) doldur
df_merged = df_merged.resample('W-FRI').last().ffill().bfill()
print(f"-> Haftalık Cuma hizalaması yapıldı. Toplam Satır: {len(df_merged)}")

# 4. Makaledeki Matematiksel Dönüşümler (Log ve Birinci Fark)
df_log_diff = np.log(df_merged).diff().dropna()
print(f"-> Logaritma ve Fark alındıktan sonra kalan Satır: {len(df_log_diff)}\n")

if len(df_log_diff) == 0:
    raise ValueError("HATA: Veri seti yine boşaldı! Lütfen yukarıdaki 'Örnek Tarih' ve 'Örnek Fiyat' çıktılarını bana kopyala, sorunun nerede olduğunu tam görelim.")

# 5. Makaledeki Min-Max Normalizasyonu (0-1)
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(df_log_diff)
df_scaled = pd.DataFrame(scaled_data, columns=df_log_diff.columns, index=df_log_diff.index)

# 6. Sütun Sıralaması (BIST100 en sona, hedef değişken olarak)
df_scaled = df_scaled[['GOLD_TL', 'VIX', 'BOND_2Y', 'BIST100']]

# 7. Makaledeki Referans Bölümleme (540 Eğitim, Kalanlar Test)
if len(df_scaled) >= 720:
    df_scaled = df_scaled.tail(720) # Son 720'yi al
    train_df = df_scaled.iloc[:540]
    test_df = df_scaled.iloc[540:720]
else:
    # 720 veri inmemişse bile sistemin çökmemesi için oransal (%75 Eğitim) bölme
    print(f"UYARI: Elimizde sadece {len(df_scaled)} hafta var. 720 eksik, oranlayarak bölüyoruz.")
    split_idx = int(len(df_scaled) * 0.75)
    train_df = df_scaled.iloc[:split_idx]
    test_df = df_scaled.iloc

ValueError: Found array with 0 sample(s) (shape=(0, 4)) while a minimum of 1 is required by MinMaxScaler.